# Parameter Golf: Sliding Window Eval + Semantic Init A/B Test

Quick A/B runs on a single Colab A100 to validate improvements before scaling to 8xH100.
Both runs use sliding window evaluation (stride=128) for better BPB.

**Runtime:** Set to **GPU > A100** (Runtime > Change runtime type)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Setup + Data (80 training shards, ~8 GB)

In [ ]:
!git clone https://github.com/openai/parameter-golf.git /content/parameter-golf
%cd /content/parameter-golf
!pip install -q sentencepiece huggingface-hub numpy
!python data/cached_challenge_fineweb.py --train-shards 10 --variant sp1024

## 2. Patch improvements (sliding window eval + semantic init)

In [ ]:
import re

src = open("train_gpt.py", "r").read()

# --- 1. Add eval_stride hyperparameter ---
src = src.replace(
    '    train_log_every = int(os.environ.get("TRAIN_LOG_EVERY", 200))',
    '    train_log_every = int(os.environ.get("TRAIN_LOG_EVERY", 200))\n'
    '    eval_stride = int(os.environ.get("EVAL_STRIDE", 0))',
)

# --- 2. Replace eval_val with sliding window version ---
NEW_EVAL_VAL = '''def eval_val(
    args: Hyperparameters,
    model: nn.Module,
    rank: int,
    world_size: int,
    device: torch.device,
    grad_accum_steps: int,
    val_tokens: Tensor,
    base_bytes_lut: Tensor,
    has_leading_space_lut: Tensor,
    is_boundary_token_lut: Tensor,
) -> tuple[float, float]:
    seq_len = args.train_seq_len
    stride = min(args.eval_stride, seq_len) if args.eval_stride > 0 else seq_len
    all_windows = val_tokens.unfold(0, seq_len + 1, stride)
    num_windows = all_windows.size(0)
    local_batch_tokens = args.val_batch_size // (world_size * grad_accum_steps)
    local_batch_seqs = max(1, local_batch_tokens // seq_len)
    win_start = (num_windows * rank) // world_size
    win_end = (num_windows * (rank + 1)) // world_size
    val_loss_sum = torch.zeros((), device=device, dtype=torch.float64)
    val_token_count = torch.zeros((), device=device, dtype=torch.float64)
    val_byte_count = torch.zeros((), device=device, dtype=torch.float64)
    model.eval()
    with torch.inference_mode():
        for batch_begin in range(win_start, win_end, local_batch_seqs):
            batch_end = min(batch_begin + local_batch_seqs, win_end)
            batch = all_windows[batch_begin:batch_end].to(device=device, dtype=torch.int64, non_blocking=True)
            x = batch[:, :-1]
            y = batch[:, 1:]
            bsz = x.size(0)
            score_mask = torch.zeros(bsz, seq_len, dtype=torch.bool, device=device)
            score_mask[:, seq_len - stride:] = True
            if batch_begin == 0:
                score_mask[0, :] = True
            y_masked = y.clone()
            y_masked[~score_mask] = -100
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=True):
                batch_loss = model(x, y_masked).detach()
            scored_count = float(score_mask.sum().item())
            val_loss_sum += batch_loss.to(torch.float64) * scored_count
            val_token_count += scored_count
            prev_flat = x.reshape(-1)
            tgt_flat = y.reshape(-1)
            mask_flat = score_mask.reshape(-1)
            token_bytes = base_bytes_lut[tgt_flat].to(dtype=torch.int16)
            token_bytes += (has_leading_space_lut[tgt_flat] & ~is_boundary_token_lut[prev_flat]).to(dtype=torch.int16)
            val_byte_count += (token_bytes.to(torch.float64) * mask_flat.to(torch.float64)).sum()
    if dist.is_available() and dist.is_initialized():
        dist.all_reduce(val_loss_sum, op=dist.ReduceOp.SUM)
        dist.all_reduce(val_token_count, op=dist.ReduceOp.SUM)
        dist.all_reduce(val_byte_count, op=dist.ReduceOp.SUM)
    val_loss = val_loss_sum / val_token_count
    bits_per_token = val_loss.item() / math.log(2.0)
    tokens_per_byte = val_token_count.item() / val_byte_count.item()
    model.train()
    return float(val_loss.item()), float(bits_per_token * tokens_per_byte)'''

old_eval = re.search(
    r'def eval_val\(.*?return float\(val_loss\.item\(\)\), float\(bits_per_token \* tokens_per_byte\)',
    src, flags=re.DOTALL,
)
assert old_eval, "Could not find eval_val function"
src = src[:old_eval.start()] + NEW_EVAL_VAL + src[old_eval.end():]

# --- 3. Add PPMI-SVD function ---
ppmi_func = '''
def build_ppmi_svd_vectors(
    shard_pattern: str, vocab_size: int, model_dim: int, window: int = 5, max_shards: int = 4,
) -> Tensor | None:
    shard_files = sorted(glob.glob(shard_pattern))[:max_shards]
    if not shard_files:
        return None
    V = vocab_size
    cooccur = np.zeros((V, V), dtype=np.float64)
    for shard_path in shard_files:
        tokens = load_data_shard(Path(shard_path)).numpy().astype(np.int64)
        n = len(tokens)
        for offset in range(1, window + 1):
            left = tokens[:n - offset]
            right = tokens[offset:]
            flat = left * V + right
            cooccur += np.bincount(flat, minlength=V * V).reshape(V, V)
            flat_rev = right * V + left
            cooccur += np.bincount(flat_rev, minlength=V * V).reshape(V, V)
    total = cooccur.sum()
    if total == 0:
        return None
    row_sums = cooccur.sum(axis=1, keepdims=True)
    col_sums = cooccur.sum(axis=0, keepdims=True)
    denom = row_sums * col_sums
    with np.errstate(divide="ignore", invalid="ignore"):
        pmi = np.where(
            (cooccur > 0) & (denom > 0),
            np.log(cooccur * total / denom),
            0.0,
        )
    ppmi = np.maximum(pmi, 0.0).astype(np.float32)
    rank = min(model_dim, V - 1)
    U, S, _ = torch.svd_lowrank(torch.from_numpy(ppmi), q=rank)
    vectors = (U * S.sqrt().unsqueeze(0)).numpy()
    if vectors.shape[1] < model_dim:
        vectors = np.concatenate(
            [vectors, np.zeros((V, model_dim - vectors.shape[1]), dtype=np.float32)], axis=1,
        )
    std = vectors.std()
    if std > 1e-6:
        vectors = vectors / std
    return torch.from_numpy(vectors)


'''

assert 'class GPT(nn.Module):' in src
src = src.replace('class GPT(nn.Module):', ppmi_func + 'class GPT(nn.Module):')

# --- 4. Add sem_embed_alpha hyperparameter ---
src = src.replace(
    '    grad_clip_norm = float(os.environ.get("GRAD_CLIP_NORM", 0.0))',
    '    grad_clip_norm = float(os.environ.get("GRAD_CLIP_NORM", 0.0))\n'
    '    sem_embed_alpha = float(os.environ.get("SEM_EMBED_ALPHA", 0.005))',
)

# --- 5. Inject sem init before training loop ---
src = src.replace(
    '    train_loader = DistributedTokenLoader(args.train_files, rank, world_size, device)\n\n    def zero_grad_all',
    '    if args.sem_embed_alpha > 0:\n'
    '        import time as _time\n'
    '        _ts = _time.perf_counter()\n'
    '        _svd = build_ppmi_svd_vectors(args.train_files, args.vocab_size, args.model_dim)\n'
    '        if _svd is not None:\n'
    '            with torch.no_grad():\n'
    '                base_model.tok_emb.weight.add_(\n'
    '                    _svd.to(dtype=base_model.tok_emb.weight.dtype, device=device) * args.sem_embed_alpha\n'
    '                )\n'
    '            log0(f"sem_embed_init:alpha={args.sem_embed_alpha} built in {1000*(_time.perf_counter()-_ts):.0f}ms")\n'
    '\n'
    '    train_loader = DistributedTokenLoader(args.train_files, rank, world_size, device)\n'
    '\n'
    '    def zero_grad_all',
)

for check in ['eval_stride', 'all_windows', 'score_mask', 'build_ppmi_svd_vectors', 'sem_embed_alpha']:
    assert check in src, f"Missing: {check}"

with open("train_gpt.py", "w") as f:
    f.write(src)

import ast
ast.parse(src)
print("All patches applied and syntax OK")

## 3. Baseline run (sliding window eval, no semantic init)

In [ ]:
!MAX_WALLCLOCK_SECONDS=180 ITERATIONS=10000 TRAIN_BATCH_TOKENS=131072 \
 VAL_LOSS_EVERY=0 TRAIN_LOG_EVERY=50 WARMUP_STEPS=10 \
 WARMDOWN_ITERS=1200 EVAL_STRIDE=128 SEM_EMBED_ALPHA=0 \
 RUN_ID=baseline python train_gpt.py

## 4. Semantic init run (sliding window eval + semantic init)

In [ ]:
!MAX_WALLCLOCK_SECONDS=180 ITERATIONS=10000 TRAIN_BATCH_TOKENS=131072 \
 VAL_LOSS_EVERY=0 TRAIN_LOG_EVERY=50 WARMUP_STEPS=10 \
 WARMDOWN_ITERS=1200 EVAL_STRIDE=128 SEM_EMBED_ALPHA=0.005 \
 RUN_ID=sem_init python train_gpt.py

## 5. Results

Fill in both `val_bpb` values from the `final_int8_zlib_roundtrip_exact` lines above.
Both runs use sliding window eval (stride=64), so these are directly comparable to each other and the leaderboard.

In [ ]:
baseline_bpb = None  # e.g. 1.4130
sem_init_bpb = None  # e.g. 1.4050

if baseline_bpb is not None and sem_init_bpb is not None:
    delta = sem_init_bpb - baseline_bpb
    direction = "WORSE" if delta > 0 else "BETTER" if delta < 0 else "SAME"
    print(f"Baseline BPB (sw64):   {baseline_bpb}")
    print(f"Semantic Init BPB:     {sem_init_bpb}")
    print(f"Delta: {delta:+.6f} ({direction})")
    print()
    print(f"Current leaderboard SOTA: 1.1194 BPB")
    print(f"Gap to SOTA: {sem_init_bpb - 1.1194:+.4f}")
    print()
    if delta < -0.001:
        print("Semantic init is helping. Consider sweeping SEM_EMBED_ALPHA.")
    elif delta < 0.001:
        print("Within noise. Try different alpha values (0.001, 0.01, 0.02).")
    else:
        print("Semantic init is hurting at full scale.")
else:
    print("Fill in both BPB values above and re-run this cell.")

## 6. Artifact size

In [ ]:
import os
for f in ["final_model.int8.ptz"]:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"{f}: {size_mb:.2f} MB")

code_bytes = len(open("train_gpt.py").read().encode())
print(f"train_gpt.py: {code_bytes / 1024:.1f} KB")

if os.path.exists("final_model.int8.ptz"):
    total = os.path.getsize("final_model.int8.ptz") + code_bytes
    print(f"Total artifact: {total / (1024*1024):.2f} MB (limit: 16 MB)")